In [ ]:
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
import numpy as np

In [ ]:
(x_train_official, y_train_official), (x_test_official, y_test_official) = cifar10.load_data()

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


In [ ]:
x_all = np.concatenate([x_train_official, x_test_official])
y_all = np.concatenate([y_train_official, y_test_official])

In [ ]:
x_all = x_all.astype('float32') / 255.0

In [ ]:
num_classes = 10
y_all = tf.keras.utils.to_categorical(y_all, num_classes)

In [ ]:
TOTAL_IMAGES = 60000
TRAIN_SIZE = 48000
VAL_SIZE = 6000

In [ ]:
x_train, y_train = x_all[:TRAIN_SIZE], y_all[:TRAIN_SIZE]
x_val, y_val = x_all[TRAIN_SIZE : TRAIN_SIZE + VAL_SIZE], y_all[TRAIN_SIZE : TRAIN_SIZE + VAL_SIZE]
x_test, y_test = x_all[TRAIN_SIZE + VAL_SIZE:], y_all[TRAIN_SIZE + VAL_SIZE:]

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

In [ ]:
INPUT_SHAPE = (32, 32, 3)

In [ ]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=INPUT_SHAPE,
    pooling='avg'
)


/tmp/ipython-input-504012206.py:2: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
x = base_model.output
predictions = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

base_model.trainable = True

In [ ]:
LEARNING_RATE = 1e-4
BATCH_SIZE = 32
EPOCHS = 10

In [ ]:
OPTIMIZER = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
LOSS_FUNCTION = 'categorical_crossentropy'

In [ ]:
model.compile(
    optimizer=OPTIMIZER,
    loss=LOSS_FUNCTION,
    metrics=['accuracy']
)

print("--- Model Summary (for Parameter Count metric) ---")
model.summary()

--- Model Summary (for Parameter Count metric) ---


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 16, 16,    │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 16, 16,    │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 16, 16,    │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 16, 16,    │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 16, 16,    │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 16, 16,    │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 16, 16,    │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 16, 16,    │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 16, 16,    │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 16, 16,    │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 16, 16,    │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 17, 17,    │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 8, 8, 96)  │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 8, 8, 96)  │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 8, 8, 96)  │          0 │ block_1_depthwis… │
│ (ReLU)              │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 8, 8, 24)  │      2,304 │ block_1_depthwis

 Total params: 2,270,794 (8.66 MB)

 Trainable params: 2,236,682 (8.53 MB)

 Non-trainable params: 34,112 (133.25 KB)

In [ ]:
import time
print(f"Starting MobileNetV2 Training for {EPOCHS} epochs...")

history = model.fit(
    x_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(x_val, y_val),
    verbose=1
)

print("\n--- Model Evaluation ---")
start_time = time.time()
loss, accuracy = model.evaluate(x_test, y_test, verbose=0)
end_time = time.time()
total_inference_time = end_time - start_time
time_per_image = total_inference_time / x_test.shape[0]
print(f"Final Test Accuracy: {accuracy*100:.2f}%")
print(f"Final Test Loss: {loss:.4f}")
print(f"Total Inference Time (6000 images): {total_inference_time:.4f} seconds")
print(f"Inference Time Per Image: {time_per_image*1000:.2f} ms")

Starting MobileNetV2 Training for 10 epochs...
Epoch 1/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 88s 24ms/step - accuracy: 0.3191 - loss: 2.0913 - val_accuracy: 0.1832 - val_loss: 2.1306
Epoch 2/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 23s 15ms/step - accuracy: 0.5487 - loss: 1.2942 - val_accuracy: 0.5800 - val_loss: 1.3774
Epoch 3/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 41s 15ms/step - accuracy: 0.6319 - loss: 1.0683 - val_accuracy: 0.6865 - val_loss: 0.9255
Epoch 4/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 22s 15ms/step - accuracy: 0.6814 - loss: 0.9263 - val_accuracy: 0.6940 - val_loss: 0.9060
Epoch 5/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - accuracy: 0.7061 - loss: 0.8494 - val_accuracy: 0.7300 - val_loss: 0.8193
Epoch 6/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 21s 14ms/step - accuracy: 0.7373 - loss: 0.7660 - val_accuracy: 0.7517 - val_loss: 0.7593
Epoch 7/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 20s 13ms/step - accuracy: 0.7549 - loss: 0.7065 - val_accuracy: 0.7622 - val_loss: 0.7384
Epoch 8/10
1500/1500 ━━━━━━━━